# Part I. ETL Pipeline for Preprocessing the Files

## Build the consolidated event data file

#### Import Python packages

In [ ]:
# Import Python packages
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv

#### Create the list of source event CSV files

In [ ]:
print(os.getcwd())
print(os.getcwd())

filepath = os.getcwd() + '/event_data'

file_path_list = []
for root, dirs, files in os.walk(filepath):
    file_path_list.extend(glob.glob(os.path.join(root, '*.csv')))

print('CSV files discovered:', len(file_path_list))

#### Create the event_data_new.csv file for Cassandra inserts

In [ ]:
full_data_rows_list = []

for f in sorted(file_path_list):
    with open(f, 'r', encoding='utf8', newline='') as csvfile:
        csvreader = csv.reader(csvfile)
        next(csvreader)

        for line in csvreader:
            full_data_rows_list.append(line)

csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_data_new.csv', 'w', encoding='utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow(['artist','firstName','gender','itemInSession','lastName','length',
                'level','location','sessionId','song','userId'])
    for row in full_data_rows_list:
        if (row[0] == ''):
            continue
        writer.writerow((row[0], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[12], row[13], row[16]))

In [ ]:
with open('event_data_new.csv', 'r', encoding='utf8') as f:
    print(sum(1 for line in f))

# Part II. Apache Cassandra Data Modeling

The `event_data_new.csv` file contains these columns:
- artist
- firstName of user
- gender of user
- item number in session
- last name of user
- length of the song
- level (paid or free song)
- location of the user
- sessionId
- song title
- userId

## Cassandra setup, query-specific tables, inserts, and verification queries

#### Creating a Cluster

In [ ]:
from cassandra.cluster import Cluster
cluster = Cluster(['127.0.0.1'])

session = cluster.connect()

#### Create Keyspace

In [ ]:
session.execute("""
CREATE KEYSPACE IF NOT EXISTS sparkify
WITH REPLICATION = {
    'class': 'SimpleStrategy',
    'replication_factor': 1
}
""")

#### Set Keyspace

In [ ]:
session.set_keyspace('sparkify')

### Create tables to support the required queries.

With Apache Cassandra, the database is modeled around the queries. This solution creates one table for each analytics question.

## Required analytics questions

### 1. Give me the artist, song title and song length in the music app history that was heard during `sessionId = 338`, and `itemInSession = 4`

### 2. Give me only the following: artist name, song title sorted by `itemInSession`, and user first and last name for `userId = 10`, `sessionId = 182`

### 3. Give me every user first and last name in my music app history who listened to the song `All Hands Against His Own`

## Query 1 table: songplays_by_session_item

In [ ]:
## Query 1: artist, song title, and length by sessionId and itemInSession
query = """
CREATE TABLE IF NOT EXISTS songplays_by_session_item (
    session_id int,
    item_in_session int,
    artist text,
    song text,
    length double,
    PRIMARY KEY (session_id, item_in_session)
)
"""
session.execute(query)

In [ ]:
file = 'event_data_new.csv'

query = """
INSERT INTO songplays_by_session_item
(session_id, item_in_session, artist, song, length)
VALUES (%s, %s, %s, %s, %s)
"""

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        session.execute(query, (int(line[8]), int(line[3]), line[0], line[9], float(line[5])))

#### Do a SELECT to verify that the data have been inserted into the table

In [ ]:
query = """
SELECT artist, song, length
FROM songplays_by_session_item
WHERE session_id = 338 AND item_in_session = 4
"""
rows = session.execute(query)
for row in rows:
    print(row.artist, row.song, row.length)

## Query 2 table: songplays_by_user_session

In [ ]:
## Query 2: artist, song, and user by userId and sessionId, sorted by itemInSession
query = """
CREATE TABLE IF NOT EXISTS songplays_by_user_session (
    user_id int,
    session_id int,
    item_in_session int,
    artist text,
    song text,
    first_name text,
    last_name text,
    PRIMARY KEY ((user_id, session_id), item_in_session)
) WITH CLUSTERING ORDER BY (item_in_session ASC)
"""
session.execute(query)

In [ ]:
query = """
INSERT INTO songplays_by_user_session
(user_id, session_id, item_in_session, artist, song, first_name, last_name)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        session.execute(query, (int(line[10]), int(line[8]), int(line[3]), line[0], line[9], line[1], line[4]))

In [ ]:
query = """
SELECT artist, song, first_name, last_name
FROM songplays_by_user_session
WHERE user_id = 10 AND session_id = 182
"""
rows = session.execute(query)
for row in rows:
    print(row.artist, row.song, row.first_name, row.last_name)

## Query 3 table: users_by_song

In [ ]:
## Query 3: users who listened to a specific song
query = """
CREATE TABLE IF NOT EXISTS users_by_song (
    song text,
    user_id int,
    first_name text,
    last_name text,
    PRIMARY KEY (song, user_id)
)
"""
session.execute(query)

In [ ]:
query = """
INSERT INTO users_by_song
(song, user_id, first_name, last_name)
VALUES (%s, %s, %s, %s)
"""

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        session.execute(query, (line[9], int(line[10]), line[1], line[4]))

In [ ]:
query = """
SELECT first_name, last_name
FROM users_by_song
WHERE song = 'All Hands Against His Own'
"""
rows = session.execute(query)
for row in rows:
    print(row.first_name, row.last_name)

### Drop the tables before closing out the sessions

In [ ]:
session.execute('DROP TABLE IF EXISTS songplays_by_session_item')
session.execute('DROP TABLE IF EXISTS songplays_by_user_session')
session.execute('DROP TABLE IF EXISTS users_by_song')

### Close the session and cluster connection

In [ ]:
session.shutdown()
cluster.shutdown()